# HR Request Routing Study — Official Colab Run

This notebook implements Topic 16 as **Qwen2.5-3B (fast SLM) → fixed escalation rules → Qwen2.5-7B (larger LLM)**. The baseline sends every request to Qwen2.5-7B. Human review is a final HR action, not a replacement for the large-model tier. Use a **T4 GPU** and run the cells in order.

## 1. Select the required hardware
In Colab choose **Runtime → Change runtime type → T4 GPU**, then continue.

In [ ]:
import os, subprocess
REPO_URL = 'https://github.com/MarieBelle88/HR-request-routing.git'
REPO_DIR = '/content/HR-request-routing'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
!pip -q install -r requirements.txt

In [ ]:
import json, platform, torch, transformers, bitsandbytes
assert torch.cuda.is_available(), 'No GPU detected. Change the Colab runtime to T4 GPU.'
gpu_name = torch.cuda.get_device_name(0)
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('Transformers:', transformers.__version__)
print('BitsAndBytes:', bitsandbytes.__version__)
print('GPU:', gpu_name)
print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
assert 'T4' in gpu_name, f'Expected a T4 GPU for the official run, found {gpu_name}'

## 2. Verify repository, labels, and tests
These checks do not run Qwen and do not produce paper results.

In [ ]:
!python scripts/verify_dataset.py
!python scripts/offline_checks.py
!python -m pytest -q

In [ ]:
import pandas as pd
records = [json.loads(line) for line in open('data/hr_requests.jsonl', encoding='utf-8')]
dev = pd.DataFrame([row for row in records if row['split'] == 'dev'])
display(dev[['request_id', 'message', 'gold_category', 'gold_urgency', 'gold_action', 'gold_escalate_to_large_model']])

## 3. Development run
This confirms model loading, JSON parsing, escalation, and evaluation on the 15 development records. Development values are not final evidence. Both quantised models are loaded sequentially.

In [ ]:
!python scripts/run_experiment.py --split dev --system both --output-dir results/dev
!python scripts/evaluate_results.py \
  --small-model results/dev/dev_small_model.jsonl \
  --baseline results/dev/dev_baseline.jsonl \
  --tiered results/dev/dev_tiered.jsonl \
  --output-dir results/dev

In [ ]:
dev_summary = json.load(open('results/dev/experiment_summary.json', encoding='utf-8'))
print('Development exact accuracy:')
print('  3B tier:', dev_summary['small_model_tier']['exact_resolution_accuracy'])
print('  Uniform 7B:', dev_summary['uniform_large_model_baseline']['exact_resolution_accuracy'])
print('  Tiered final:', dev_summary['tiered_system']['exact_resolution_accuracy'])
print('Development escalation F1:', dev_summary['tiered_system']['large_model_escalation_f1'])

## 4. Locked official test run
Do not change the prompt, rules, threshold, code, or labels after starting this cell. The following 45 held-out records provide the values used in the paper.

In [ ]:
!python scripts/run_experiment.py --split test --system both --output-dir results
!python scripts/evaluate_results.py \
  --small-model results/test_small_model.jsonl \
  --baseline results/test_baseline.jsonl \
  --tiered results/test_tiered.jsonl \
  --output-dir results

In [ ]:
summary = json.load(open('results/experiment_summary.json', encoding='utf-8'))
systems = {
    '3B tier': summary['small_model_tier'],
    'Uniform 7B': summary['uniform_large_model_baseline'],
    'Tiered final': summary['tiered_system'],
}
display(pd.DataFrame(systems).loc[[
    'category_accuracy', 'urgency_accuracy', 'action_accuracy',
    'exact_resolution_accuracy', 'human_review_f1',
    'small_model_calls', 'large_model_calls',
    'total_gpu_seconds', 'peak_vram_bytes'
]])
comparison = summary['comparison']
print('7B-call reduction:', f"{comparison['large_model_call_reduction']:.1%}")
print('Relative cost-proxy reduction:', f"{comparison['relative_cost_proxy_reduction']:.1%}")
print('GPU-seconds reduction:', f"{comparison['gpu_seconds_reduction']:.1%}")
print('Tiered escalation precision/recall/F1:',
      summary['tiered_system']['large_model_escalation_precision'],
      summary['tiered_system']['large_model_escalation_recall'],
      summary['tiered_system']['large_model_escalation_f1'])

In [ ]:
runtime = json.load(open('results/runtime_environment.json', encoding='utf-8'))
display(pd.Series(runtime, name='recorded value').to_frame())

## 5. Download genuine result files
Download this ZIP and add its contents to the repository before finalising the paper.

In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive('/content/HR_Routing_Official_Results', 'zip', root_dir=REPO_DIR, base_dir='results')
print('Created:', archive)
files.download(archive)